# Pipeline con dos modelos para predecir

In [77]:
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

# ---------- Cargar modelo y datos normalizados
model_1 = load_model("../../models/model_7_6/model_7_6.keras")
model_2 = load_model("../../models/model_7_9/model_7_9.keras")

with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)


In [78]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [79]:
with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)

In [80]:
df_model = df_data.drop(columns=[
    "ride_id",
    "ended_at",
    "time_hms_ms",
    "member_casual",
    "start_station_id",
    "end_station_id",
    "month",
    "day",
    "temperature",
    "wind_speed",
    "precipitation",
    "relative_humidity",
    "snow_depth",
    "hour_float",
    "rideable_type_classic_bike",
    "rideable_type_electric_bike",
    "rideable_type_docked_bike",
    "member_casual",
    "member_casual_bool",
    "duration_min",
    #"started_at"
])

# Se cambia la precision por minutos

In [81]:
# Se redonde al minuto mas cercano
df_model["started_minute"] = df_model["started_at"].dt.round("min")

In [82]:
df_agg = df_model.groupby([
    "start_station_idx",
    "end_station_idx",
    "started_minute"
]).agg(
    #n_viajes=("ride_id", "count"),
    year=("year", "first"),
    temp_std=("temp_std", "first"),
    wind_std=("wind_std", "first"),
    rel_humidity_std=("rel_humidity_std", "first"),
    precipitation_std=("precipitation_std", "first"),
    snow_depth_std=("snow_depth_std", "first"),
    hour_sin=("hour_sin", "first"),
    hour_cos=("hour_cos", "first"),
    month_sin=("month_sin", "first"),
    month_cos=("month_cos", "first"),
    event=("event", "any"),  # True si al menos un dato es true
    normal_day=("day_type_Normal", "any"),  # True si al menos un dato es true
    weekend_day=("day_type_Weekend", "any"),  # True si al menos un dato es true
    holiday_day=("day_type_Holiday", "any"),  # True si al menos un dato es true
    #member_casual=("member_casual_bool", "any"),  # True si al menos un dato es true
    #classic_bike=("rideable_type_classic_bike", "any"),  # True si al menos un dato es true
    #docked_bike=("rideable_type_docked_bike", "any"),  # True si al menos un dato es true
    #electric_bike=("rideable_type_electric_bike", "any"),  # True si al menos un dato es true
    #duration_min_mean=("duration_min", "mean"),
).reset_index()

In [83]:
df_agg.shape

(9073410, 17)

# Se obtiene los datos

In [ ]:
x_context = df_agg.drop(columns=[
    "start_station_idx",
    "end_station_idx",
    "started_minute"
])

x_start = df_agg[[
    "start_station_idx",
]]

x_end = df_agg[[
    "end_station_idx",
]]

#y_real = df_agg["n_viajes_agg"]

KeyError: 'n_viajes_agg'

In [ ]:
x_context.shape

(9073410, 14)

In [ ]:
x_context.dtypes

year                   int64
temp_std             float64
wind_std             float64
rel_humidity_std     float64
precipitation_std    float64
snow_depth_std       float64
hour_sin             float64
hour_cos             float64
month_sin            float64
month_cos            float64
event                   bool
normal_day              bool
weekend_day             bool
holiday_day             bool
dtype: object

# Se realiza la primera predicción

In [ ]:
# Predicciones como probabilidades
y_predictions_prob = model_1.predict({
    'start_station': x_start,
    'end_station': x_end,
    'context': x_context
}, batch_size=256)

print("Shape de predicciones:", y_predictions_prob.shape)

35444/35444 ━━━━━━━━━━━━━━━━━━━━ 27s 761us/step
Shape de predicciones: (9073410, 8)


In [ ]:
# La clase 1 (1 viaje) se compone de la clase 0 a la 3

max_prob_class_1 = np.max(y_predictions_prob[:, 0:4], axis=1)

In [ ]:
target = 8671457  # número de viajes reales clase 1 en los datos disponibles

# Ordenas de mayor a menor
sorted_probs = np.sort(max_prob_class_1)[::-1]

# El umbral es el valor en la posición target - 1
threshold = sorted_probs[target - 1]

print("Umbral:", threshold)

Umbral: 0.22306772


In [ ]:
y_predictions_class_1 = np.where(max_prob_class_1 >= threshold, True, False)

# Contabilizar los 1 en y_predictions_1
count_1 = np.sum(y_predictions_class_1 == True)
count_1

np.int64(8671457)

# Se realiza la segunda predicción

In [ ]:
df_agg['predict_class_1'] = y_predictions_class_1

In [ ]:
df_agg.head()

,start_station_idx,end_station_idx,started_minute,year,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_sin,hour_cos,month_sin,month_cos,event,normal_day,weekend_day,holiday_day,predict_class_1
0,0,0,2022-02-11 13:38:00,2022,-1.379839,1.021427,1.326397,-0.099704,-0.056148,-0.416215,-0.909266,0.866025,5.000000e-01,False,False,False,True,True
1,0,0,2022-03-05 15:03:00,2022,-0.612834,-0.276355,-0.807078,-0.099704,-0.056148,-0.717569,-0.696487,1.000000,6.123234e-17,False,False,True,False,True
2,0,0,2022-03-06 10:55:00,2022,-1.156238,4.113864,0.687365,-0.099704,-0.056148,0.281155,-0.959662,1.000000,6.123234e-17,False,False,True,False,True
3,0,0,2022-03-20 17:06:00,2022,-0.424108,-0.883141,-1.019270,-0.099704,-0.056148,-0.972183,-0.234223,1.000000,6.123234e-17,False,False,True,False,True
4,0,0,2022-03-20 17:11:00,2022,-0.410327,-0.825775,-1.038403,-0.099704,-0.056148,-0.976781,-0.214238,1.000000,6.123234e-17,False,False,True,False,True


In [ ]:
df_agg = df_agg[df_agg['predict_class_1'] == False]

In [ ]:
df_agg.head()

,start_station_idx,end_station_idx,started_minute,year,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_sin,hour_cos,month_sin,month_cos,event,normal_day,weekend_day,holiday_day,predict_class_1
48117,17,14,2024-05-25 21:21:00,2024,0.422517,0.162661,-1.439539,-0.099704,-0.056148,-0.639215,0.769028,0.5,-0.866025,False,False,True,False,False
55109,42,1771,2022-07-30 21:42:00,2022,1.204149,-0.707578,-1.211030,-0.099704,-0.056148,-0.565207,0.824949,-0.5,-0.866025,True,False,True,False,False
55113,42,1789,2022-07-09 20:09:00,2022,0.863062,1.247069,-0.969525,-0.099704,-0.056148,-0.845067,0.534660,-0.5,-0.866025,True,False,True,False,False
55342,44,1414,2024-05-12 21:36:00,2024,1.142133,0.843266,-1.736277,-0.099704,-0.056148,-0.589373,0.807861,0.5,-0.866025,False,False,True,False,False
56047,50,1860,2022-07-16 19:42:00,2022,0.801047,0.849328,0.601268,-0.099704,-0.056148,-0.903366,0.428869,-0.5,-0.866025,True,False,True,False,False


In [ ]:
df_agg.shape

(401953, 18)

# Se obtienen los datos

In [ ]:
x_context = df_agg.drop(columns=[
    "start_station_idx",
    "end_station_idx",
    "started_minute",
    "predict_class_1"
])

x_start = df_agg[[
    "start_station_idx",
]]

x_end = df_agg[[
    "end_station_idx",
]]

In [ ]:
x_context.shape

(401953, 14)

In [ ]:
x_context.dtypes

year                   int64
temp_std             float64
wind_std             float64
rel_humidity_std     float64
precipitation_std    float64
snow_depth_std       float64
hour_sin             float64
hour_cos             float64
month_sin            float64
month_cos            float64
event                   bool
normal_day              bool
weekend_day             bool
holiday_day             bool
dtype: object

In [ ]:
y_pred = model_2.predict({
    'start_station': x_start,
    'end_station': x_end,
    'context': x_context
})

12562/12562 ━━━━━━━━━━━━━━━━━━━━ 7s 530us/step


In [ ]:
y_pred

array([[2.2544603],
       [2.2020755],
       [2.1409101],
       ...,
       [1.9042685],
       [1.8892856],
       [1.9378017]], shape=(401953, 1), dtype=float32)

In [ ]:
y_pred_int = np.rint(y_pred).astype(int)  # redondear al entero más cercano
y_pred_int

array([[2],
       [2],
       [2],
       ...,
       [2],
       [2],
       [2]], shape=(401953, 1))

Se muestran cuantos datos hay de cada tipo

In [85]:
import numpy as np

unique, counts = np.unique(y_pred_int, return_counts=True)
print(unique)
print(counts)


[2 3 4 5]
[387595  10611   3611    136]


Los datos correctos son

2 -> 372173

3 -> 24990

4 ->  4098

5 -> 692